In [ ]:
import torch

In [ ]:
# Kernel: интерпретатор `.venv` проекта (после `uv sync` пакет ставится в editable-режиме).
import torch

from image_transformation.methods.gaf_transformation import GAF
from image_transformation.methods.mtf_transformation import MTF
from image_transformation.methods.stft_transformation import STFTSpectrogram

print("torch", torch.__version__)
print("MTF / GAF / STFT — импорты ок")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

assert "MTF" in globals(), "Сначала выполните ячейку 1 (sys.path + импорты MTF, GAF, STFTSpectrogram)."

# Один ряд длины 61 — как фикстура `data` в test_mtf.py / test_stft.py (rng=42)
rng = np.random.default_rng(42)
x_1d = rng.normal(size=(61,)).astype(np.float64)
X = torch.as_tensor(x_1d, dtype=torch.float64).unsqueeze(0)

print("Вход X (batch, time):", tuple(X.shape))

# Параметры по аналогии с test_gaf.py
gaf = GAF({"method": "summation", "overlapping": True, "image_size": 0.7})
y_gaf = gaf.transform(X)
print("GAF выход (batch, H, W):", tuple(y_gaf.shape))

# Параметры как в test_mtf.py (quantile, overlapping=True)
mtf = MTF(
    {
        "image_size": 0.7,
        "n_bins": 8,
        "strategy": "quantile",
        "overlapping": True,
        "flatten": False,
    }
)
y_mtf = mtf.transform(X)
print("MTF выход (batch, H, W):", tuple(y_mtf.shape))

# Параметры как в test_stft.py (window hann, center=False)
stft = STFTSpectrogram(
    {
        "window_size": 16,
        "hop_length": 4,
        "n_fft": 32,
        "window_type": "hann",
        "center": False,
        "power": 2.0,
        "normalized": False,
    }
)
y_stft = stft.transform(X)
print("STFT выход (batch, freq_bins, frames):", tuple(y_stft.shape))

# --- графики ---
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
ax0, ax1, ax2, ax3 = axes.ravel()

ax0.plot(x_1d)
ax0.set_title(f"Исходный ряд, длина T = {x_1d.shape[0]}")
ax0.set_xlabel("t")
ax0.set_ylabel("x(t)")

gaf_np = y_gaf[0].detach().cpu().numpy()
im1 = ax1.imshow(gaf_np, aspect="equal", cmap="viridis")
ax1.set_title(f"GAF (GASF)\nвход {tuple(X.shape)} → выход {tuple(y_gaf.shape)}")
fig.colorbar(im1, ax=ax1, fraction=0.046)

mtf_np = y_mtf[0].detach().cpu().numpy()
im2 = ax2.imshow(mtf_np, aspect="equal", cmap="magma")
ax2.set_title(f"MTF\nвход {tuple(X.shape)} → выход {tuple(y_mtf.shape)}")
fig.colorbar(im2, ax=ax2, fraction=0.046)

stft_np = y_stft[0].detach().cpu().numpy()
im3 = ax3.imshow(stft_np, aspect="auto", origin="lower", cmap="inferno")
ax3.set_title(
    f"STFT (power)\nвход {tuple(X.shape)} → выход {tuple(y_stft.shape)}"
)
ax3.set_xlabel("frame (ось времени)")
ax3.set_ylabel("частотный bin")
fig.colorbar(im3, ax=ax3, fraction=0.046)

fig.suptitle(
    "Один и тот же ряд: три преобразования (параметры по образцу test_gaf / test_mtf / test_stft)",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
plt.show()